# Измерение площадей и расстояний


Корректность расчётов **расстояний**, **площадей** и построения **буферных зон** напрямую зависит от используемой **системы координат (CRS)**.

Если данные находятся в **географической системе координат** (например, **WGS 84 (EPSG:4326)**), координаты выражены в **градусах**. Такие данные подходят для **отображения объектов на карте**, но **не подходят для прямых измерений расстояний и площадей**.

Чтобы получать результаты в **метрах** и **квадратных метрах**, данные необходимо перепроецировать в подходящую **проецированную систему координат**, например **UTM**.

В этом разделе мы рассмотрим:

- почему перед выполнением измерений важно **перепроецировать данные** и как **CRS влияет на результаты измерений**;
- как вычислять **площади полигонов**;
- как измерять **расстояния между объектами**.


## 0. Импорт библиотек и подготовка данных


### 0.1 Импорт библиотек


In [20]:
import osmnx as ox
import geopandas as gpd

from shapely.geometry import Point

### 0.2 Подготовка данных


Загрузим границу района из OpenStreetMap


In [2]:
area_name = "Центральный район, Санкт-Петербург"

admin_border = ox.geocode_to_gdf(area_name)

In [3]:
# Альтернатива: загрузка из сохранённого файла (если osmnx недоступен)
# Скачать файл osm_data.gpkg: https://drive.google.com/file/d/1uhnwDvDwZHhUUum04cp7okzWcvVoQ43Y/view?usp=sharing

# admin_border = gpd.read_file("../data/osm_data.gpkg", layer="central_district")

Проверим систему координат:


In [4]:
admin_border.crs

<Geographic 2D CRS: EPSG:4326>
Name: WGS 84
Axis Info [ellipsoidal]:
- Lat[north]: Geodetic latitude (degree)
- Lon[east]: Geodetic longitude (degree)
Area of Use:
- name: World.
- bounds: (-180.0, -90.0, 180.0, 90.0)
Datum: World Geodetic System 1984 ensemble
- Ellipsoid: WGS 84
- Prime Meridian: Greenwich

Загрузим станции метро в выбранном районе из OpenStreetMap


In [5]:
tags = {"railway": "subway_entrance"}

metro = ox.features_from_place(
    area_name,
    tags
)

metro.explore()

In [6]:
# Альтернатива: загрузка из сохранённого файла (если osmnx недоступен)
# Скачать файл osm_data.gpkg: https://drive.google.com/file/d/1uhnwDvDwZHhUUum04cp7okzWcvVoQ43Y/view?usp=sharing

# metro = gpd.read_file("../data/osm_data.gpkg", layer="metro_entrances")

Проверим систему координат:


In [7]:
metro.crs

<Geographic 2D CRS: EPSG:4326>
Name: WGS 84
Axis Info [ellipsoidal]:
- Lat[north]: Geodetic latitude (degree)
- Lon[east]: Geodetic longitude (degree)
Area of Use:
- name: World.
- bounds: (-180.0, -90.0, 180.0, 90.0)
Datum: World Geodetic System 1984 ensemble
- Ellipsoid: WGS 84
- Prime Meridian: Greenwich

## 1. Измерение площади

В GeoPandas каждый `GeoDataFrame` имеет специальный атрибут `.geometry`, в котором хранятся **геометрические объекты** пространственных данных — например, точки, линии или полигоны.

Этот атрибут позволяет обращаться к геометрии объектов и выполнять различные **пространственные операции**.

Одной из таких операций является вычисление площади. Для этого используется атрибут `.area`, который возвращает **площадь каждого полигонального объекта**.

Важно помнить, что `.area` возвращает результат **в тех единицах измерения, которые используются в системе координат (CRS)**.


### 1.1 Вычисление площади 1.0

Создадим новое поле `"area"` в `GeoDataFrame` `admin_border` и запишем в него вычисленные значения площади.


In [21]:
admin_border['area'] = admin_border.geometry.area

/var/folders/ry/9bb7wrz54vq_kn2ytlj6ynzm0000gn/T/ipykernel_97524/1329868239.py:1: UserWarning: Geometry is in a geographic CRS. Results from 'area' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  admin_border['area'] = admin_border.geometry.area


Посмотрим на результат


In [9]:
admin_border[['name', 'area']]

,name,area
0,Tsentralny District,0.002842


На этом этапе можно заметить, что полученные значения площади оказываются **неожиданно малыми**. Это связано с тем, что слой всё ещё находится в **географической системе координат**.

В географических системах координат (например, **WGS 84 / EPSG:4326**) положение объектов задаётся в **градусах широты и долготы**, а не в линейных единицах измерения. Поэтому площадь в таком случае вычисляется в **квадратных градусах**.

Такие значения сложно интерпретировать, поскольку **градусы не являются единицей длины**, а их пространственный масштаб **изменяется в зависимости от широты**.

Поэтому для корректного вычисления площадей и расстояний данные необходимо предварительно **перепроецировать** в систему координат с метрическими единицами измерения, например **UTM**.

> **Обратите внимание на предупреждение `UserWarning`**, которое может появляться при вычислении площади.
> Оно указывает на то, что расчёты выполняются для данных в **географической системе координат**, и полученные значения могут быть некорректными. Такое предупреждение служит напоминанием о необходимости **перепроецировать данные**.


### 1.2 Перепроецирование данных в UTM

Перепроецируем данные в подходящую **UTM-зону**, чтобы координаты были выражены в **метрах**.

Сначала определим подходящую систему координат UTM с помощью метода `.estimate_utm_crs()`. И перепроецируем слой `admin_border` в эту систему координат.


In [10]:
utm_crs = admin_border.estimate_utm_crs()

admin_border_utm = admin_border.to_crs(utm_crs)

Убедимся, что система координат действительно изменилась


In [11]:
admin_border_utm.crs

<Projected CRS: EPSG:32636>
Name: WGS 84 / UTM zone 36N
Axis Info [cartesian]:
- E[east]: Easting (metre)
- N[north]: Northing (metre)
Area of Use:
- name: Between 30°E and 36°E, northern hemisphere between equator and 84°N, onshore and offshore. Belarus. Cyprus. Egypt. Ethiopia. Finland. Israel. Jordan. Kenya. Lebanon. Moldova. Norway. Palestine. Russian Federation. Saudi Arabia. Sudan. Syria. Türkiye (Turkey). Uganda. Ukraine.
- bounds: (30.0, 0.0, 36.0, 84.0)
Coordinate Operation:
- name: UTM zone 36N
- method: Transverse Mercator
Datum: World Geodetic System 1984 ensemble
- Ellipsoid: WGS 84
- Prime Meridian: Greenwich

Мы видим, что в этой системе координат используются метры. Теперь вычислим значения площади ещё раз.


### 1.3 Вычисление площади 2.0


Создадим новое поле `"area_m2"` в `GeoDataFrame` `admin_border_utm` и запишем туда вычисленные значения площади.


In [12]:
admin_border_utm['area_m2'] = admin_border_utm.geometry.area

Посмотрим на результат


In [13]:
admin_border_utm[['name', 'area', 'area_m2']]

,name,area,area_m2
0,Tsentralny District,0.002842,1.769497e+07


Теперь полученные значения выглядят более реалистично. Поскольку площадь объектов вычисляется в **квадратных метрах**.

Иногда значения площади могут отображаться в **экспоненциальной записи** (scientific notation). Такой формат используется для компактного представления больших чисел.

Число вида `a.bcde+X` означает $a.bcde \times 10^{X}$.

Например:
`1.769497e+07` = $1.769497 \times 10^{7}$ или 17,7 млн.

Это просто альтернативная форма записи большого числа.


Для удобства можно перевести площадь в квадратные километры (1 км² = 1 000 000 м²):


In [14]:
admin_border_utm["area_km2"] = admin_border_utm["area_m2"] / 1_000_000
admin_border_utm[["display_name", "area_km2"]].head()

,display_name,area_km2
0,"Tsentralny District, Литейный округ, Saint Pet...",17.694966


## 2. Измерение расстояний

Метод `.distance()` позволяет вычислять расстояние между геометрическими объектами.
Предположим, что мы хотим определить, на каком расстоянии находятся станции метро от центра района.

Сделаем это дважды, как и с площадью: сначала на исходных данных в **градусах**, затем после **перепроецирования в UTM**. На этот раз ошибка окажется интереснее.


### 2.0. Подготовка данных


Сначала перепроецируем слой со станциями метро в ту же систему координат, что и границы района.


In [15]:
metro_utm = metro.to_crs(utm_crs)

Также определим центр района, который будем использовать как точку отсчёта для измерения расстояний. Он понадобится нам в обеих системах координат, поскольку измерение мы выполним дважды.


In [23]:
center = admin_border_utm.geometry.centroid.iloc[0]          # центр района, UTM

print(center)

POINT (352577.928183038 6647178.876633392)


In [24]:
center_gdf = gpd.GeoDataFrame(geometry=[center], crs=utm_crs)
center_deg = center_gdf.to_crs("EPSG:4326").geometry.iloc[0]  # тот же центр, EPSG:4326

center_gdf.explore()

Центроид — это геометрический центр полигона.
Мы используем его как условную точку центра района.


### 2.1. Вычисление расстояний 1.0

Сначала выполним измерение на исходных данных — и станции, и центр района заданы в EPSG:4326, то есть в градусах.


In [17]:
metro["distance_deg"] = metro.geometry.distance(center_deg)

metro[["name", "distance_deg"]].sort_values("distance_deg").head()

/var/folders/ry/9bb7wrz54vq_kn2ytlj6ynzm0000gn/T/ipykernel_97524/437645754.py:1: UserWarning: Geometry is in a geographic CRS. Results from 'distance' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  metro["distance_deg"] = metro.geometry.distance(center_deg)


name  distance_deg
element id                                             
node    318455877       Площадь Восстания      0.004119
        318455870       Площадь Восстания      0.004253
        1109163132                    NaN      0.005818
        1109163144                    NaN      0.005863
        1109163066  Площадь Восстания - 2      0.006261

Маленькие числа без единиц измерения — та же проблема, что и с площадью в квадратных градусах. Но с расстояниями в градусах есть проблема серьёзнее: **результат зависит от направления.**

Градус широты почти везде одинаков — около 111 км. Градус долготы уменьшается к полюсам в cos(широты) раз — это тот же множитель, который растягивает Web Mercator в [первом разделе](projections_1.ipynb). На широте Санкт-Петербурга (около 60°) градус долготы примерно вдвое короче градуса широты.

Поэтому две точки, одинаково удалённые на местности, могут дать разные «расстояния» в градусах. Возьмём две точки ровно в километре от центра района — одну строго на восток, другую строго на север:


In [18]:
east = Point(center.x + 1000, center.y)    # 1 км на восток от центра
north = Point(center.x, center.y + 1000)   # 1 км на север от центра

pair_utm = gpd.GeoSeries([center, east, north], crs=utm_crs)
pair_deg = pair_utm.to_crs("EPSG:4326")

for label, i in [("восток", 1), ("север", 2)]:
    print(f"{label:>6}: {pair_utm[0].distance(pair_utm[i]):7.1f} м"
          f" | {pair_deg[0].distance(pair_deg[i]):.6f} градусов")

восток:  1000.0 м | 0.017878 градусов
 север:  1000.0 м | 0.008998 градусов


В метрах оба расстояния равны километру, как и должно быть. В градусах расстояние на восток примерно **в 2 раза** больше, чем на север, — снова 1 / cos(60°).

Градус — не единица длины. Измеряя в градусах, мы искажаем не только величину расстояния, но и делаем её зависимой от направления, поэтому никакой пересчёт такой результат не исправит.


### 2.2. Вычисление расстояний 2.0

Теперь выполним то же измерение на перепроецированных слоях.


In [19]:
metro_utm["distance_m"] = metro_utm.geometry.distance(center)

metro_utm[["name", "distance_m"]].sort_values("distance_m").head()

name  distance_m
element id                                       
node    318455870   Площадь Восстания  452.068225
        318455877   Площадь Восстания  452.720277
        4060185664         Маяковская  589.744361
        1109163128         Маяковская  594.497554
        1109163132                NaN  646.838040

Полученные значения показывают расстояние в метрах, поскольку данные находятся в проецированной системе координат UTM.


> **Евклидово расстояние**
>
> Метод `.distance()` вычисляет **евклидово расстояние** между геометриями — то есть **прямое расстояние по плоскости** между двумя точками.
>
> Такое расстояние иногда называют **«расстоянием по прямой»** (_straight-line distance_ или _as-the-crow-flies_).
>
> В реальных задачах часто используют **расстояния по улично-дорожной сети** — например, по дорогам или маршрутам общественного транспорта.  
> С методами сетевого анализа мы познакомимся в одном из следующих модулей курса.
>
> Тем не менее для многих аналитических задач **евклидово расстояние является полезным приближением**, особенно при предварительном анализе пространственных данных.


## Итог


В этом разделе мы рассмотрели, как **корректно измерять площади и расстояния** с помощью GeoPandas.

Мы узнали:

- почему географические CRS не подходят для измерений — градусы искажают не только величину расстояния, но и делают её зависимой от направления;
- как вычислять площади с помощью `.area`;
- как измерять расстояния с помощью `.distance()`;

Корректный выбор системы координат — одно из ключевых этапов подготовки данных для последующего пространственного анализа.


> **Перед выполнением пространственных измерений проверьте:**
>
> - находится ли слой в **проецированной CRS**;
> - выражены ли координаты в **метрах**;
> - совпадает ли **CRS у всех используемых слоёв**.
